# Phase 0 — Data Survey: Brazilian Retail Investor Reddit Study

**Scope:** r/investimentos · r/farialimabets  
**Source:** Watchful1 separated-subreddit dump, Academic Torrents, coverage 2005-06 to 2024-12  
**Goal:** Verify sampling frame viability before Phase 1. No LLM calls, no clustering.

All random seeds = 42. Files are streamed — never fully loaded into memory.

In [ ]:
import json
import random
import datetime
import warnings
from pathlib import Path
from collections import defaultdict

import zstandard as zstd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from langdetect import detect, LangDetectException

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

SEED = 42
random.seed(SEED)

BASE = Path('/home/theoriffel/codes/decade-ba-case')
RAW  = BASE / 'data/raw/reddit/subreddits24'
INT  = BASE / 'data/interim'
REP  = BASE / 'reports'
FIG  = REP / 'figures'

FILES = {
    'investimentos': {
        'submissions': RAW / 'investimentos_submissions.zst',
        'comments':    RAW / 'investimentos_comments.zst',
    },
    'farialimabets': {
        'submissions': RAW / 'farialimabets_submissions.zst',
        'comments':    RAW / 'farialimabets_comments.zst',
    },
}

STUDY_WINDOWS = {
    'A': ('2020-06', '2021-06'),
    'B': ('2022-01', '2023-01'),
    'C': ('2024-01', '2024-12'),
}

def stream_zst(path):
    dctx = zstd.ZstdDecompressor(max_window_size=2**31)
    with open(path, 'rb') as fh:
        with dctx.stream_reader(fh) as reader:
            buf = b''
            while True:
                chunk = reader.read(1 << 20)
                if not chunk:
                    break
                buf += chunk
                lines = buf.split(b'\n')
                buf = lines[-1]
                for line in lines[:-1]:
                    line = line.strip()
                    if line:
                        try:
                            yield json.loads(line)
                        except json.JSONDecodeError:
                            pass
            if buf.strip():
                try:
                    yield json.loads(buf)
                except json.JSONDecodeError:
                    pass

def utc_to_ym(ts):
    try:
        return datetime.datetime.fromtimestamp(int(ts), datetime.UTC).strftime('%Y-%m')
    except Exception:
        return None

def utc_to_date(ts):
    try:
        return datetime.datetime.fromtimestamp(int(ts), datetime.UTC).strftime('%Y-%m-%d %H:%M UTC')
    except Exception:
        return None

print('Environment ready.')

## Task 2 — File Inventory

In [ ]:
inventory = json.loads((INT / 'file_inventory.json').read_text())

rows = []
for key, v in inventory.items():
    rows.append({
        'File': Path(v['path']).name,
        'Size (MB)': v['size_mb'],
        'Records': f"{v['total_records']:,}",
        'Earliest': v['earliest_utc'],
        'Latest': v['latest_utc'],
        '[deleted]/AutoModerator': f"{v['deleted_or_automod']:,}",
    })

df_inv = pd.DataFrame(rows)
display(df_inv)

## Task 3 — Monthly Volume

In [ ]:
from IPython.display import Image, display as ipy_display

print('Submission volume:')
ipy_display(Image(str(FIG / 'monthly_volume_submissions.png')))

print('Comment volume:')
ipy_display(Image(str(FIG / 'monthly_volume_comments.png')))

## Task 4 — Window Feasibility Table

In [ ]:
df_feas = pd.read_csv(INT / 'window_feasibility.csv')

def highlight_verdict(val):
    if 'AT RISK' in str(val):
        return 'background-color: #ffcdd2'
    elif 'MARGINAL' in str(val):
        return 'background-color: #fff9c4'
    elif 'VIABLE' in str(val):
        return 'background-color: #c8e6c9'
    return ''

display(df_feas.style.applymap(highlight_verdict, subset=['verdict']))

at_risk = df_feas[df_feas['submissions_ge3_comments'] < 300]
if len(at_risk):
    print(f'WARNING: {len(at_risk)} cells below n=300 threshold:')
    display(at_risk)
else:
    print('All 6 cells are VIABLE (≥300 submissions with ≥3 comments). No flagged strata.')

## Task 5 — Author-Thread Linkage Check

In [ ]:
linkage = json.loads((INT / 'linkage_check.json').read_text())

for sub, r in linkage.items():
    print(f'r/{sub}')
    print(f'  Sampled submissions : {r["sampled_submissions"]:,}')
    print(f'  Matched (≥1 comment): {r["matched_submissions"]:,} ({r["pct_matched"]}%)')
    print(f'  Mean comments/thread: {r["mean_comments_per_matched"]}')
    print(f'  Median comments/thrd: {r["median_comments_per_matched"]}')
    pcts = r['top_comment_score_percentiles']
    print(f'  Comment score dist  : p10={pcts["p10"]} p25={pcts["p25"]} p50={pcts["p50"]} p75={pcts["p75"]} p90={pcts["p90"]}')
    print()

## Task 6 — Raw Sample Inspection (Window C, 2024)

In [ ]:
raw_samples = json.loads((INT / 'raw_samples_window_c.json').read_text())

for sub, posts in raw_samples.items():
    print('=' * 70)
    print(f'  r/{sub} — 10 random posts from Window C (2024-01 to 2024-12)')
    print('=' * 70)
    for i, p in enumerate(posts, 1):
        ts = int(p.get('created_utc', 0))
        date = datetime.datetime.fromtimestamp(ts, datetime.UTC).strftime('%Y-%m-%d') if ts else 'N/A'
        title = p.get('title', '(no title)')
        st = (p.get('selftext', '') or '')[:600]
        score = p.get('score', 0)
        nc = p.get('num_comments', 0)
        print(f'\n[{i}] {date} | score={score} | comments={nc}')
        print(f'TITLE: {title}')
        print(f'BODY : {st if st else "(empty / link post)"}')
        print('-' * 60)
    print()

## Task 7 — Quality Scan

In [ ]:
qual = json.loads((INT / 'quality_scan.json').read_text())

# Summary table
rows = []
for sub, q in qual.items():
    rows.append({
        'Subreddit': f'r/{sub}',
        'Total submissions': f"{q['total_submissions']:,}",
        'Deleted %': q['selftext_deleted']['pct'],
        'Removed %': q['selftext_removed']['pct'],
        'Empty %': q['selftext_empty']['pct'],
        'Link posts %': q['link_posts']['pct'],
        'Bot authors %': q['bot_authors']['pct'],
        'Text len median': q['selftext_length']['median'],
        'Text len mean': q['selftext_length']['mean'],
    })
display(pd.DataFrame(rows))

print()
print('Language distribution (langdetect on 1,000-post sample):')
lang_rows = []
for sub, q in qual.items():
    for lang, pct in list(q['language_distribution_pct'].items())[:8]:
        lang_rows.append({'Subreddit': f'r/{sub}', 'Language': lang, 'Pct': pct})
display(pd.DataFrame(lang_rows).pivot(index='Language', columns='Subreddit', values='Pct').fillna(0.0))

In [ ]:
# Quality bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cats = ['Deleted %', 'Removed %', 'Empty %', 'Link posts %', 'Bot authors %']
colors = ['#ef9a9a', '#ffcc80', '#fff176', '#90caf9', '#ce93d8']

for ax, (sub, q) in zip(axes, qual.items()):
    vals = [
        q['selftext_deleted']['pct'],
        q['selftext_removed']['pct'],
        q['selftext_empty']['pct'],
        q['link_posts']['pct'],
        q['bot_authors']['pct'],
    ]
    bars = ax.barh(cats, vals, color=colors)
    ax.set_xlabel('% of total submissions')
    ax.set_title(f'r/{sub}', fontweight='bold')
    ax.set_xlim(0, 75)
    for bar, val in zip(bars, vals):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val}%', va='center', fontsize=9)

plt.suptitle('Submission Quality Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG / 'quality_scan.png', dpi=150, bbox_inches='tight')
plt.show()

## Task 8 — Summary

Full summary written to `reports/phase0_summary.md`. Key verdicts inline:

In [ ]:
from IPython.display import Markdown
summary_md = (REP / 'phase0_summary.md').read_text()
display(Markdown(summary_md))